# Vizualizacija društvenih mreža — layouti, vizualne varijable i workflow

**Poglavlje:** [7. Vizualizacija društvenih mreža](../content/07_vizualizacija_mreza.md)

Ova bilježnica prati tekst poglavlja 7 i dodaje konkretne primjere koji se mogu pokretati i mijenjati.

**Što ćemo proći:**

1. **Sirovi prikaz** — referenca kako „ništa ne reći”.
2. **Pet layouta** (spring, kamada-kawai, circular, shell, spectral) na istoj mreži.
3. **Veličina = degree centralnost**.
4. **Usporedba 4 mjere centralnosti** kao veličine čvora (degree / betweenness / closeness / eigenvector) — pedagoški ključna.
5. **Boja = zajednica (Louvain)**.
6. **Ego-mreža** — kako se „zumirati” u jednog čvora.
7. **Težinske mreže** — debljina brida = težina.
8. **Usporedba layouta** na težinskoj mreži.
9. **Anti-primjer „hairball”** i kako ga popraviti.
10. **Povijesna mreža** — firentinske obitelji (Medici kao paradigma betweennessa).
11. **Reproducibilnost** — važnost `seed`-a.
12. **Publikacijska kvaliteta** — 300 dpi + SVG + legenda + izvor.
13. **Accessibility** — color-blind friendly paleta i checklist.
14. **Export za Gephi** — GEXF/GraphML workflow.
15. **Interaktivna vizualizacija** (pyvis, opcionalno).

Koristimo klasične dataset-e: **Zachary karate club**, **Florentine families** (oba ugrađeni u NetworkX) i malu sintetičku **mrežu razgovora u razredu**.

## 0. Setup

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from networkx.algorithms import community

plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10

def scale_sizes(values, lo=80, hi=1200):
    """Linearno reskalira vrijednosti u raspon [lo, hi] za veličine čvorova."""
    arr = np.array(list(values), dtype=float)
    if arr.max() == arr.min():
        return [ (lo+hi)/2 ] * len(arr)
    return list(lo + (hi - lo) * (arr - arr.min()) / (arr.max() - arr.min()))

## 1. Sirovi prikaz — što vidimo bez ikakvih odluka?

Počinjemo s najjednostavnijim pozivom `nx.draw(G)`. Cilj: imati referentnu točku koliko sirovi prikaz ne komunicira.

In [ ]:
G = nx.karate_club_graph()
print(f'Čvorova: {G.number_of_nodes()}, bridova: {G.number_of_edges()}')

plt.figure(figsize=(6, 5))
nx.draw(G, with_labels=True, node_color='lightgray', node_size=300)
plt.title('1. Sirovi prikaz — default layout, bez odluka')
plt.axis('off')
plt.show()

Prikaz nam kaže: graf postoji, ima ~34 čvora, gust je. Ali **tko je važan**, **ima li grupa**, **tko su mostovi** — ne znamo. Sada ćemo slojevito dodati informaciju.

## 2. Različiti layouti — ista mreža, pet priča

Istu mrežu prikazujemo u 5 layouta. **Ključno**: `seed=42` fiksira početnu nasumičnost (reproducibilnost). Bez toga svaki run izgleda drukčije.

In [ ]:
layouts = {
    'spring (force-directed)': nx.spring_layout(G, seed=42),
    'kamada-kawai': nx.kamada_kawai_layout(G),
    'circular': nx.circular_layout(G),
    'shell': nx.shell_layout(G),
    'spectral': nx.spectral_layout(G),
}

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
for ax, (name, pos) in zip(axes, layouts.items()):
    nx.draw(G, pos, ax=ax, node_color='steelblue', node_size=120,
            edge_color='lightgray', with_labels=False)
    ax.set_title(name)
    ax.axis('off')
plt.suptitle('Ista mreža, pet layouta — koja priča?', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

**Kako čitati:**
- **Spring / Kamada-Kawai**: odmah vide se dvije jezgre (Mr. Hi i Officer) — force-directed layout ih grupira.
- **Circular**: svi čvorovi imaju „jednako pravo glasa”, ali strukturu zajednica uopće ne vidimo.
- **Shell**: koncentrični krugovi, koristan kad postoji jezgra-periferija; ovdje nije idealno.
- **Spectral**: projekcija u 2D kroz Laplacijana; često „rastegne” graf po prvoj glavnoj strukturi.

Za SNA u 95% slučajeva → **spring** ili **kamada-kawai**.

## 3. Vizualna varijabla: veličina = degree centralnost

Kodirajmo **degree centralnost** u veličinu čvora. Odmah će se „vidjeti” tko je hub.

In [ ]:
pos = nx.spring_layout(G, seed=42)
deg = dict(G.degree())
sizes = [deg[n] * 60 + 80 for n in G.nodes()]

plt.figure(figsize=(9, 7))
nx.draw_networkx_edges(G, pos, alpha=0.3)
nx.draw_networkx_nodes(G, pos, node_size=sizes,
                       node_color='steelblue', alpha=0.85)
top5 = sorted(deg, key=deg.get, reverse=True)[:5]
labels = {n: f'{n}\n(deg={deg[n]})' for n in top5}
nx.draw_networkx_labels(G, pos, labels=labels, font_size=9, font_weight='bold')
plt.title('Veličina čvora = degree centralnost\n(labele samo za top 5)')
plt.axis('off')
plt.tight_layout()
plt.show()

print('Top 5 po degree-u:')
for n in top5:
    print(f'  Čvor {n}: degree = {deg[n]}')

## 4. KLJUČNO: usporedba 4 mjere centralnosti

Različite centralnosti mjere različite aspekte „važnosti” (pogl. 3). **Vizualizacija ih čini uspoređivima**: pokažimo ISTI graf, ali svaki put veličinu čvora kodira druga mjera. Pratimo: tko je veliki u kojoj mjeri?

In [ ]:
centralities = {
    'Degree\n(tko ima najviše veza)': nx.degree_centrality(G),
    'Betweenness\n(tko je "most")': nx.betweenness_centrality(G),
    'Closeness\n(tko je blizu svima)': nx.closeness_centrality(G),
    'Eigenvector\n(tko je "uz važne")': nx.eigenvector_centrality(G, max_iter=1000),
}

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
for ax, (name, cent) in zip(axes, centralities.items()):
    sizes_c = scale_sizes([cent[n] for n in G.nodes()], lo=50, hi=1400)
    nx.draw_networkx_edges(G, pos, alpha=0.25, ax=ax)
    nx.draw_networkx_nodes(G, pos, node_size=sizes_c,
                           node_color=[cent[n] for n in G.nodes()],
                           cmap='plasma', alpha=0.9,
                           edgecolors='white', linewidths=1.2, ax=ax)
    top3 = sorted(cent, key=cent.get, reverse=True)[:3]
    top_labels = {n: n for n in top3}
    nx.draw_networkx_labels(G, pos, labels=top_labels,
                            font_size=10, font_weight='bold', ax=ax)
    ax.set_title(name, fontsize=11)
    ax.axis('off')

plt.suptitle('Karate klub — 4 mjere centralnosti kao veličina čvora',
             y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

print('\nTop 3 po svakoj mjeri:')
for name, cent in centralities.items():
    t3 = sorted(cent, key=cent.get, reverse=True)[:3]
    print(f'  {name.splitlines()[0]:15s} → {t3}')

**Pedagoški ključ:** obratite pažnju da **nisu isti čvorovi najveći u svim mjerama**.
- **Degree i eigenvector** se slažu — instruktor (0) i predsjednik (33).
- **Betweenness** može izdvojiti „mostove” (npr. čvor koji je u sredini puteva između dvije grupe), a ne nužno hubove.
- **Closeness** često istakne one blizu centra mreže.

Ista mreža → **četiri različite priče o „moći”**. Odabir mjere = interpretativna odluka.

## 5. Vizualna varijabla: boja = zajednica (Louvain)

Pokrenut ćemo Louvain (pogl. 6) i bojom kodirati pripadnost zajednici. Kombinirano s veličinom = degree dobijamo dvostruko informativnu sliku.

In [ ]:
comms = community.louvain_communities(G, seed=42)
print(f'Broj zajednica: {len(comms)}')

node2comm = {n: i for i, c in enumerate(comms) for n in c}
colors = [node2comm[n] for n in G.nodes()]

plt.figure(figsize=(10, 7))
nx.draw_networkx_edges(G, pos, alpha=0.3)
nx.draw_networkx_nodes(G, pos,
                       node_size=sizes,
                       node_color=colors,
                       cmap='Set2', alpha=0.9,
                       edgecolors='white', linewidths=1.5)
nx.draw_networkx_labels(G, pos, labels=labels, font_size=9, font_weight='bold')

Q = community.modularity(G, comms)
plt.title(f'Karate klub — veličina = degree, boja = Louvain zajednica\nModularnost Q = {Q:.3f}')
plt.axis('off')
plt.tight_layout()
plt.show()

for i, c in enumerate(comms):
    print(f'Zajednica {i}: {sorted(c)}')

## 6. Ego-mreža: zumiranje u jednog čvora

Kad je cijela mreža prevelika ili kad želimo razumjeti **neposredno okruženje** jedne osobe, koristimo **ego-mrežu**: čvor + svi njegovi susjedi + veze među njima. Ovo je **klasična tehnika za portret jednog aktera** (studenti je koriste za case-studies pojedinaca).

In [ ]:
ego_center = 0
ego = nx.ego_graph(G, ego_center, radius=1)
print(f'Ego-mreža čvora {ego_center}: {ego.number_of_nodes()} čvorova, {ego.number_of_edges()} veza')

pos_e = nx.spring_layout(ego, seed=7)

node_colors = ['crimson' if n == ego_center else 'lightsteelblue' for n in ego.nodes()]
node_sizes = [1200 if n == ego_center else 450 for n in ego.nodes()]

plt.figure(figsize=(9, 7))
nx.draw_networkx_edges(ego, pos_e, alpha=0.5, edge_color='gray')
nx.draw_networkx_nodes(ego, pos_e, node_size=node_sizes, node_color=node_colors,
                       edgecolors='white', linewidths=2)
nx.draw_networkx_labels(ego, pos_e, font_size=9, font_weight='bold')
plt.title(f'Ego-mreža čvora {ego_center} (Mr. Hi) — radius=1\n'
          f'{ego.number_of_nodes()-1} direktnih kontakata, {ego.number_of_edges()} veza ukupno')
plt.axis('off')
plt.tight_layout()
plt.show()

ego_density = nx.density(ego)
print(f'Gustoća ego-mreže: {ego_density:.3f}')
print('(Visoka gustoća = prijatelji Mr. Hi-ja se međusobno poznaju = „zatvoreni” ego)')

**Gustoća ego-mreže** je Burtov koncept (pogl. 4): visoka = closed ego (svi se znaju, bonding kapital); niska = open ego (ego je broker, bridging kapital). Ovo je praktičan portret jedne osobe u kulturnom/socijalnom kontekstu.

## 7. Težinske mreže — debljina brida = težina

Mala **mreža razgovora u razredu** gdje težina brida = broj razgovora u tjednu.

In [ ]:
H = nx.Graph()
razgovori = [
    ('Ana', 'Bruno', 8), ('Ana', 'Carla', 10), ('Bruno', 'Carla', 9),
    ('Carla', 'David', 3), ('David', 'Eva', 7), ('David', 'Filip', 6),
    ('Eva', 'Filip', 8), ('Filip', 'Goran', 2),
    ('Goran', 'Hana', 9), ('Goran', 'Ivan', 8), ('Hana', 'Ivan', 7),
]
H.add_weighted_edges_from(razgovori)

pos_h = nx.spring_layout(H, seed=7, k=0.8)
weights = [H[u][v]['weight'] for u, v in H.edges()]
widths = [w * 0.5 for w in weights]

comms_h = community.louvain_communities(H, seed=42)
n2c_h = {n: i for i, c in enumerate(comms_h) for n in c}
colors_h = [n2c_h[n] for n in H.nodes()]

plt.figure(figsize=(10, 7))
nx.draw_networkx_edges(H, pos_h, width=widths, alpha=0.5, edge_color='gray')
nx.draw_networkx_nodes(H, pos_h, node_size=900,
                       node_color=colors_h, cmap='Pastel1',
                       edgecolors='black', linewidths=1.5)
nx.draw_networkx_labels(H, pos_h, font_size=10, font_weight='bold')
edge_labels = {(u, v): H[u][v]['weight'] for u, v in H.edges()}
nx.draw_networkx_edge_labels(H, pos_h, edge_labels=edge_labels, font_size=8)
plt.title('Mreža razgovora u razredu\nbroj na bridu = razgovori/tjedan, boja = zajednica')
plt.axis('off')
plt.tight_layout()
plt.show()

Vide se tri grupe prijateljstava, povezane **tankim bridovima** (slabe veze). Carla i Filip su **brokeri** preko rijetkih, ali važnih veza (Granovetter).

## 8. Usporedba layouta na istoj težinskoj mreži

In [ ]:
layouts_h = {
    'spring (force-directed)': nx.spring_layout(H, seed=7),
    'kamada-kawai': nx.kamada_kawai_layout(H),
    'circular': nx.circular_layout(H),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (name, p) in zip(axes, layouts_h.items()):
    nx.draw_networkx_edges(H, p, width=widths, alpha=0.5, edge_color='gray', ax=ax)
    nx.draw_networkx_nodes(H, p, node_size=700,
                           node_color=colors_h, cmap='Pastel1',
                           edgecolors='black', linewidths=1.2, ax=ax)
    nx.draw_networkx_labels(H, p, font_size=9, ax=ax)
    ax.set_title(name)
    ax.axis('off')
plt.suptitle('Isti podaci, tri layouta — circular gubi strukturu zajednica', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 9. Anti-primjer: „hairball” i kako ga popraviti

In [ ]:
Big = nx.barabasi_albert_graph(500, 3, seed=42)
pos_b = nx.spring_layout(Big, seed=42)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

nx.draw(Big, pos_b, ax=ax1, node_size=20, node_color='steelblue',
        edge_color='lightgray', alpha=0.6, with_labels=False)
ax1.set_title(f'ANTI-PRIMJER: hairball\n({Big.number_of_nodes()} čvorova, ništa se ne vidi)')
ax1.axis('off')

deg_b = dict(Big.degree())
top_nodes = sorted(deg_b, key=deg_b.get, reverse=True)[:50]
Sub = Big.subgraph(top_nodes)
pos_sub = nx.spring_layout(Sub, seed=42, k=0.6)
sizes_sub = [deg_b[n] * 2 for n in Sub.nodes()]

comms_sub = community.louvain_communities(Sub, seed=42)
n2c_sub = {n: i for i, c in enumerate(comms_sub) for n in c}
colors_sub = [n2c_sub[n] for n in Sub.nodes()]

nx.draw_networkx_edges(Sub, pos_sub, alpha=0.3, ax=ax2)
nx.draw_networkx_nodes(Sub, pos_sub, node_size=sizes_sub,
                       node_color=colors_sub, cmap='tab10',
                       alpha=0.9, edgecolors='white', linewidths=1.2, ax=ax2)
ax2.set_title('RJEŠENJE: top 50 po degree-u + boja = zajednica\n(informativno i čitljivo)')
ax2.axis('off')

plt.tight_layout()
plt.show()

## 10. Povijesni primjer — Firentinske obitelji i uspon Medicia

**Najpoznatiji primjer u SNA-u**: Padgett & Ansell (1993) analizirali su mrežu brakova između firentinskih obitelji 15. stoljeća. Mediči **nisu imali najviše brakova** (degree nije najveći!), ali su imali **najviši betweenness** — bili su **most** između blokova obitelji. To objašnjava njihov politički uspon (nadmoć kroz strukturalnu poziciju, ne kroz masovnost).

Ovo je klasična lekcija: **različite mjere centralnosti otkrivaju različite oblike moći**.

In [ ]:
F = nx.florentine_families_graph()
print(f'Obitelji: {F.number_of_nodes()}, brakovi: {F.number_of_edges()}')

deg_f = dict(F.degree())
bet_f = nx.betweenness_centrality(F)

pos_f = nx.kamada_kawai_layout(F)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 7))

sizes_deg = scale_sizes([deg_f[n] for n in F.nodes()], lo=200, hi=1800)
nx.draw_networkx_edges(F, pos_f, alpha=0.4, ax=ax1)
nx.draw_networkx_nodes(F, pos_f, node_size=sizes_deg,
                       node_color=[deg_f[n] for n in F.nodes()],
                       cmap='Blues', edgecolors='black', linewidths=1.2, ax=ax1)
nx.draw_networkx_labels(F, pos_f, font_size=9, ax=ax1)
top_deg = max(deg_f, key=deg_f.get)
ax1.set_title(f'Veličina = DEGREE (broj brakova)\nNajvišu degree ima: {top_deg} (deg={deg_f[top_deg]})',
              fontsize=11)
ax1.axis('off')

sizes_bet = scale_sizes([bet_f[n] for n in F.nodes()], lo=200, hi=1800)
nx.draw_networkx_edges(F, pos_f, alpha=0.4, ax=ax2)
nx.draw_networkx_nodes(F, pos_f, node_size=sizes_bet,
                       node_color=[bet_f[n] for n in F.nodes()],
                       cmap='Reds', edgecolors='black', linewidths=1.2, ax=ax2)
nx.draw_networkx_labels(F, pos_f, font_size=9, ax=ax2)
top_bet = max(bet_f, key=bet_f.get)
ax2.set_title(f'Veličina = BETWEENNESS (strukturalna moć)\nNajviši betweenness ima: {top_bet} ({bet_f[top_bet]:.3f})',
              fontsize=11)
ax2.axis('off')

plt.suptitle('Firentinske obitelji 15. st. (Padgett & Ansell, 1993) — dvije priče o moći',
             y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

print('\nUsporedba ranga (top 5):')
print(f'  Po degree:      {sorted(deg_f, key=deg_f.get, reverse=True)[:5]}')
print(f'  Po betweenness: {sorted(bet_f, key=bet_f.get, reverse=True)[:5]}')

**Poanta:** Mediči nisu najveći po degree-u, ali **dominiraju po betweennessu** — njihov politički uspon (uspon Cosima de' Medici u ranom 15. st.) može se **strukturalno objasniti**: oni su bili **most** između inače nepovezanih obiteljskih blokova.

**Lekcija za kulturalnog studenta:** kad pišete o moći, popularnosti ili utjecaju u kulturi (glumac, umjetnik, influencer, politički akter) — birajte mjeru koja odgovara vašem argumentu i **uvijek objasnite zašto ta mjera**. Veličina čvora na slici je argument, ne neutralna činjenica.

## 11. Reproducibilnost: važnost `seed`-a

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, ax in enumerate(axes):
    p = nx.spring_layout(G)
    nx.draw(G, p, ax=ax, node_color='steelblue', node_size=100,
            edge_color='lightgray', with_labels=False)
    ax.set_title(f'spring_layout bez seed-a, pokušaj {i+1}')
    ax.axis('off')
plt.suptitle('BEZ seed-a: tri izvršavanja, tri različite slike iste mreže', y=1.02)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, ax in enumerate(axes):
    p = nx.spring_layout(G, seed=42)
    nx.draw(G, p, ax=ax, node_color='seagreen', node_size=100,
            edge_color='lightgray', with_labels=False)
    ax.set_title(f'spring_layout(seed=42), pokušaj {i+1}')
    ax.axis('off')
plt.suptitle('SA seed=42: tri izvršavanja, ISTA slika — reproducibilno', y=1.02)
plt.tight_layout()
plt.show()

## 12. Publikacijska kvaliteta — s istaknutim brokerima

Finalna vizualizacija kombinira: dobar layout, veličinu = degree, boju = zajednica, **istaknute bridging čvorove** (top po betweennessu → deblja bijela kontura), selektivne oznake, legendu, naslov, izvor.

In [ ]:
pos = nx.spring_layout(G, seed=42, k=0.5, iterations=60)
bet_g = nx.betweenness_centrality(G)
top_brokers = sorted(bet_g, key=bet_g.get, reverse=True)[:3]

edgecolors = ['crimson' if n in top_brokers else 'white' for n in G.nodes()]
linewidths_n = [3.5 if n in top_brokers else 1.5 for n in G.nodes()]

fig, ax = plt.subplots(figsize=(11, 8))
nx.draw_networkx_edges(G, pos, alpha=0.25, edge_color='#888', ax=ax)
nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=colors,
                       cmap='Set2', alpha=0.92,
                       edgecolors=edgecolors, linewidths=linewidths_n, ax=ax)

label_nodes = set(top5) | set(top_brokers)
nx.draw_networkx_labels(G, pos,
                        labels={n: n for n in label_nodes},
                        font_size=10, font_weight='bold', ax=ax)

legend_elements = []
for i, c in enumerate(comms):
    color = cm.Set2(i / max(len(comms)-1, 1))
    legend_elements.append(plt.Line2D([0], [0], marker='o', color='w',
                          markerfacecolor=color, markersize=10,
                          label=f'Zajednica {i+1} (n={len(c)})'))
legend_elements.append(plt.Line2D([0], [0], marker='o', color='w',
                      markerfacecolor='lightgray', markeredgecolor='crimson',
                      markeredgewidth=2, markersize=11,
                      label='Top-3 broker (betweenness)'))
ax.legend(handles=legend_elements, loc='upper left', frameon=True, fontsize=9)

ax.set_title('Zachary Karate Club — struktura zajednica i brokeri\n'
             f'Louvain, {len(comms)} zajednice, Q = {Q:.3f}. '
             'Veličina = degree, crvena kontura = top-3 broker.',
             fontsize=12, pad=14)
ax.text(0.01, -0.03,
        'Izvor: Zachary (1977). Layout: spring (seed=42). Alat: NetworkX + matplotlib.',
        transform=ax.transAxes, fontsize=8, color='gray')
ax.axis('off')
plt.tight_layout()

plt.savefig('karate_publikacija.png', dpi=300, bbox_inches='tight')
plt.savefig('karate_publikacija.svg', bbox_inches='tight')
print('Spremljeno: karate_publikacija.png (300 dpi) i karate_publikacija.svg')
plt.show()

## 13. Accessibility — color-blind friendly paleta i checklist

Oko 8% muškaraca i 0.5% žena ima neki oblik daltonizma. **Defaultne palete nisu uvijek dostupne.** Koristimo:
- **Sekvencijalne podatke** (centralnost, težina) → `viridis`, `cividis`, `plasma` (percepcijski jednolike + color-blind safe).
- **Kategorije** (zajednica) → `Set2`, `Dark2`, `tab10` (ColorBrewer kvalitativne palete).
- **Izbjegavati**: default `jet`, sirove crveno/zelene kombinacije, previše nijansi jedne boje.

In [ ]:
palettes = ['jet', 'viridis', 'cividis', 'Set2']
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, pal in zip(axes, palettes):
    nx.draw_networkx_edges(G, pos, alpha=0.25, ax=ax)
    nx.draw_networkx_nodes(G, pos, node_size=sizes,
                           node_color=colors, cmap=pal,
                           alpha=0.9, edgecolors='white', linewidths=1.2, ax=ax)
    ax.set_title(f"cmap='{pal}'" + ('  (LOŠE)' if pal == 'jet' else '  (OK)'),
                 fontsize=11)
    ax.axis('off')
plt.suptitle('Usporedba paleta — "jet" nije color-blind safe i nije percepcijski jednolika',
             y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

print('\nChecklist za svaku publikacijsku vizualizaciju:')
for item in [
    '[ ] Fiksirani seed (reproducibilno)',
    '[ ] Legenda za sve vizualne varijable',
    '[ ] Naslov koji kaže ŠTO i KAKO',
    '[ ] Izvor podataka naveden',
    '[ ] Color-blind friendly paleta',
    '[ ] Max 5-7 kategorija u boji (ostalo "ostali")',
    '[ ] Labele samo za važne čvorove (ne sve)',
    '[ ] Export u SVG ili PNG 300 dpi',
    '[ ] Anonimizacija ako su podaci o pojedincima',
]:
    print('  ' + item)

## 14. Export za Gephi — profesionalni workflow

U akademskom/publikacijskom radu često kombiniramo: **analiza u Pythonu** (reproducibilno) → **finala stilizacija u Gephi-ju** (interaktivno, ForceAtlas2, filtriranje). NetworkX eksportira u **GEXF** (Gephi native) i **GraphML** (univerzalni).

Ako dodate atribute (centralnosti, zajednica) kao node attributes, oni se u Gephi-ju direktno pojave kao kontrolni parametri za boju/veličinu/filter.

In [ ]:
G_export = G.copy()
for n in G_export.nodes():
    G_export.nodes[n]['degree'] = deg[n]
    G_export.nodes[n]['betweenness'] = round(bet_g[n], 4)
    G_export.nodes[n]['community'] = node2comm[n]
    G_export.nodes[n]['label'] = f'Node_{n}'

nx.write_gexf(G_export, 'karate_za_gephi.gexf')
nx.write_graphml(G_export, 'karate.graphml')
print('Spremljeno:')
print('  karate_za_gephi.gexf  → otvoriti u Gephi-ju (File > Open)')
print('  karate.graphml        → univerzalno (Cytoscape, yEd, Gephi)')
print('\nNode atributi dostupni u Gephi-ju: degree, betweenness, community, label')

**Workflow:**
1. U Pythonu: očistiti podatke, izračunati mjere, pokrenuti Louvain, spremiti kao GEXF.
2. U Gephi-ju: otvoriti GEXF, pokrenuti ForceAtlas2 layout, bojati po `community`, veličinu po `betweenness`, filtrirati po `degree > X`, eksportirati sliku.
3. U članku: dokumentirati oba koraka (mjera + vizualna odluka).

## 15. Interaktivna vizualizacija (opcionalno) — pyvis

Za web-objave i eksploraciju, `pyvis` generira HTML s zoomom, drag-om i hoverom. Raskomentirajte ako imate `pyvis` instaliran (`pip install pyvis`).

In [ ]:
# from pyvis.network import Network
# net = Network(height='600px', width='100%', notebook=True, cdn_resources='in_line')
# for n in G.nodes():
#     net.add_node(int(n), label=str(n),
#                  size=10 + deg[n]*2,
#                  group=int(node2comm[n]),
#                  title=f'Node {n}\ndegree={deg[n]}\nbetweenness={bet_g[n]:.3f}')
# for u, v in G.edges():
#     net.add_edge(int(u), int(v))
# net.show('karate_interactive.html')
# print('Otvorite karate_interactive.html u pregledniku.')

## Sažetak

- Layout **nije neutralan**: spring/kamada-kawai pokazuju zajednice, circular ih skriva.
- **Veličina = centralnost, boja = zajednica, debljina brida = težina** — standardna trojka.
- **Različite mjere centralnosti = različite priče o moći** (sekcija 4 i Mediči u sekciji 10).
- **Ego-mreža** je praktičan portret jednog aktera.
- Uvijek fiksirajte **seed** — reproducibilnost je znanstveni standard.
- Za velike mreže **filtrirajte** (top-N) ili **agregirajte** (zajednice kao super-čvorovi).
- **Accessibility**: izbjegavajte `jet`, koristite `viridis`/`cividis`/`Set2`.
- **Export u GEXF** spaja Python-analizu s Gephi-jevom vizualnom finom-tuningom.
- Svaka publikacijska slika treba: naslov, legendu, izvor, ime layouta.

Nastavak: [8. Difuzija i utjecaj](../content/08_difuzija_inovacija_utjecaj.md)